<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/Learning-Material/05-Introduction%20to%20LangChain/Module-01/1.3_WebSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web Search tool in LLMs
## using Tavily API

In [35]:
# Clone the repo
! git clone --depth 1 https://github.com/langchain-ai/lca-lc-foundations.git

Cloning into 'lca-lc-foundations'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 140 (delta 27), reused 85 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.09 MiB | 8.21 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [3]:
!pip install -q langchain_groq tavily

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00


In [11]:
from google.colab import userdata
from langchain.messages import HumanMessage
from langchain_groq import ChatGroq
from tavily import TavilyClient
from langchain.agents import create_agent
from langchain.tools import tool

## Setting up Tavily

In [12]:
tavily_api_key = userdata.get('tavily_key')
tavily_client = TavilyClient(tavily_api_key)

In [39]:
@tool
def web_search(query: str) -> str:
    """Searches the web for the given input query."""
    response = tavily_client.search(query)
    if response.get('answer'):
        return response['answer']
    elif 'results' in response:
        # Concatenate content of top results as a fallback
        return " ".join([r['content'] for r in response['results'] if 'content' in r][:3])
    else:
        return "No relevant information found."

## Set LLM

In [37]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get('groq_key'),
    temperature=0.1,
)

## Create Agent

In [40]:
agent = create_agent(
    model=llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [41]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='0bfa1868-07ce-4bbc-b5b2-e6c9f4327baf'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'gk8fpdbp9', 'function': {'arguments': '{"query":"current mayor of San Francisco"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 227, 'total_tokens': 246, 'completion_time': 0.056448028, 'completion_tokens_details': None, 'prompt_time': 0.011828433, 'prompt_tokens_details': None, 'queue_time': 0.173825153, 'total_time': 0.068276461}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd90c-5d2f-7671-9a26-292c87bf4f66-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'gk8fpdbp9', 'type': 'tool_call'}